In [48]:
import sys
sys.path.append('..')
import torch
import soundfile as sf
import json
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
import numpy as np
import argparse
import random

from encoder import FMEncoder, compute_spectrogram_cqt
from fm_ddsp import fm_renderer, make_mod_matrix
from nnAudio.features import CQT2010v2
from generate_dataset import generate_dataset
from train import train_stage1

%load_ext autoreload
%autoreload 2

# Stage 1 — Single modulator, single carrier
    # Fixed f0=440, one modulator → one carrier, vary ratio of modulator and level. Encoder learns the relationship between ratio and sideband position.
# Stage 2 — Two operators, fixed algorithm
    # Still fixed f0, introduce a second modulator. Encoder learns to handle multiple simultaneous modulators.
# Stage 3 — Variable f0, fixed algorithm
    # Now vary f0 across MIDI range. Encoder learns pitch invariance — same timbre at different pitches should give same parameters.
# Stage 4 — Multiple algorithms
    # Introduce algorithm variety. Encoder learns to distinguish different routing topologies.
# Stage 5 — Full dataset
    # Everything varying simultaneously.

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [49]:
def stage1_params():
    ratio_choices = [0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0]
    # only use operator 2
    ratio_op2 = random.choice(ratio_choices)
    mod_level_op2 = torch.rand(1)
    return {
        'f0': 110.0,
        'algorithm': 'algo_2',
        'mod_values': torch.tensor([0,0,0,0,0,0,1], dtype=torch.float32),
        'ratios': torch.tensor([1,1,ratio_op2,1], dtype = torch.float32),
        'levels': torch.tensor([0,0,mod_level_op2, 1], dtype = torch.float32),
        'carrier_weights': torch.tensor([0.0, 0.0, 0.0, 1.0], dtype = torch.float32)
    }

In [50]:
save_dir_pc = "B:\\TrainingData\\stage1_1"
#save_dir = '../data/stage1',
args = argparse.Namespace(
    n_examples = 40000,
    save_dir = save_dir_pc,
    Fs = 16000,
    duration = 1.0,
    seed = 127,
    overwrite=True
)
generate_dataset(args, param_fn = stage1_params)

Low pass filter created, time used = 0.0000 seconds
num_octave =  7
No early downsampling is required, downsample_factor =  1
Early downsampling filter created,                         time used = 0.0000 seconds
CQT kernels created, time used = 0.0056 seconds
Generating example 00000/40000
Generating example 04000/40000
Generating example 08000/40000
Generating example 12000/40000
Generating example 16000/40000
Generating example 20000/40000
Generating example 24000/40000
Generating example 28000/40000
Generating example 32000/40000
Generating example 36000/40000
Generating example 39999/40000
Finished Generating Examples


In [47]:
# train stage 1
args = argparse.Namespace(
    data_dir = save_dir_pc,
    Fs = 16000,
    duration = 1.0,
    lr = 0.0001,
    n_epochs = 30,
    resume = None,
    start_epoch = 0
)
train_stage1(args)

Using Cuda? True
Low pass filter created, time used = 0.0000 seconds
num_octave =  7
No early downsampling is required, downsample_factor =  1
Early downsampling filter created,                         time used = 0.0000 seconds
CQT kernels created, time used = 0.0020 seconds
Epoch 0/30:


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x0000028B6812AF20>
Traceback (most recent call last):
  File "C:\Users\Marcus\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py", line 1618, in __del__
    self._shutdown_workers()
  File "C:\Users\Marcus\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py", line 1582, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "C:\Users\Marcus\AppData\Local\Programs\Python\Python312\Lib\multiprocessing\process.py", line 149, in join
    res = self._popen.wait(timeout)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Marcus\AppData\Local\Programs\Python\Python312\Lib\multiprocessing\popen_spawn_win32.py", line 112, in wait
    res = _winapi.WaitForSingleObject(int(self._handle), msecs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt: 


KeyboardInterrupt: 